In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

with open('carteisan.txt','r',encoding='utf-8') as f:
    text=f.read()
    char = sorted(set(text))
    print(char)
    vocab_size=len(char)
    

cuda
['\n', ' ', '!', '#', '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '4', '5', '6', '7', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'œ', '—', '‘', '’', '“', '”']


In [4]:
print(len(char))

83


In [18]:
batch_size = 32   # Câte secvențe procesăm în paralel
block_size = 8
learning_rate = 1e-3   # Rata de învățare (1e-3 sau 1e-2 pentru Bigram)
eval_iters=250
max_iters = 1000
dropout=0.2
n_embd=384
n_layer=4
embedding_vector={0.1,0.2}
string_to_int = { ch:i for i,ch in enumerate(char) }
int_to_string = {i:ch for i,ch in enumerate(char) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])
encoded_hello=encode('hello')
decode_hello=decode(encoded_hello)
data = torch.tensor(encode(text),dtype = torch.long)
print(data[:100])
n= int(0.8*len(data))
train_data=data[:n]
val_data=data[n:]
def get_batch(split):
    data= train_data if split == 'train' else val_data
    ix= torch.randint(len(data)- block_size, (batch_size,))
    print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y= torch.stack([data[i+1:i+block_size+1] for i in ix])
    x,y =x.to(device), y.to(device)
    return x,y

    x,y = get_batch('train')
    print('inputs:')
    print(x)
    print('targets:')
    print(y)

tensor([ 0, 41, 59, 70, 62, 55, 19,  1, 41, 58, 55,  1, 63, 55, 54, 54, 62, 55,
        68,  0,  0, 22, 71, 70, 58, 65, 68, 19,  1, 44,  9,  1, 24,  9,  1, 41,
        71, 70, 70, 62, 55,  0,  0, 30, 62, 62, 71, 69, 70, 68, 51, 70, 65, 68,
        19,  1, 43,  9,  1, 26,  9,  1, 37, 75, 62, 55, 69,  0,  0,  0,  1,  1,
         1,  1,  1,  1,  1,  1,  0, 39, 55, 62, 55, 51, 69, 55,  1, 54, 51, 70,
        55, 19,  1, 31, 71, 62, 75,  1, 13, 18])


In [13]:
@torch.no_grad()
def estimate_loss():
    out={}
    model.eval()
    for split in ['train','val']:
        losses=torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits, loss =model(X,Y)
            losses[k]=loss.item()
        out[split]=losses.mean()
    model.train()
    return out

In [14]:
block_size=8
x =train_data[:block_size]
y=train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target=y[t]
    print('when input is', context, 'target is', target)

when input is tensor([0]) target is tensor(41)
when input is tensor([ 0, 41]) target is tensor(59)
when input is tensor([ 0, 41, 59]) target is tensor(70)
when input is tensor([ 0, 41, 59, 70]) target is tensor(62)
when input is tensor([ 0, 41, 59, 70, 62]) target is tensor(55)
when input is tensor([ 0, 41, 59, 70, 62, 55]) target is tensor(19)
when input is tensor([ 0, 41, 59, 70, 62, 55, 19]) target is tensor(1)
when input is tensor([ 0, 41, 59, 70, 62, 55, 19,  1]) target is tensor(41)


In [19]:
class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self,head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float ('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size = num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    """ a simple linear layer folowed by non-linearity """

    def __init__(self, n_embd)
        super().__init_()
        self.net =nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4* n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self,x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    
    def ___init__(self,n_embd,n_head):
        super().__init__()
        head_size=n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self,x):
        y=self.sa(x)
        x=self.ln1(x+y)
        y=self.ffwd(x)
        x=self.ln2(x+y)
        return x
        

class GPTLanguageModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks=nn.Sequental(*(Block(n_embd,n_head=n_head) for _ in range(n_layer)))

        self.ln_f=nn.LayerNorm(n_embd)
        self.lm_head=nn.Linear(n_embd,vocab_size)

        self.apply(self._init_weights)

    def __init_weights(self,module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.2)
            if module.bias is not None:
                torch.nn.init.zeros_(module_bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index,targets=None):
           B, T = index.shape

            tok_emb =self.token_embedding_table(idx)
            pos_emb=self.position_embedding_table(torch.arrange(T,device=device)
            X=tok_emb+pos_emb
            x=self.blocks(x)
            x=self.ln_f(x)
            logits=self.lm_head(x)


        
            if targets is None:
                loss=None
            else:
                B,T,C = logits.shape
                logits =logits.view(B*T,C)
                targets= targets.view(B*T)
                loss = F.cross_entropy(logits,targets)
            
            return logits,loss

    def generate(self,index,max_new_tokens):
            for _ in range(max_new_tokens):
                logits, loss =self.forward(index)
                logits=logits[:,-1,:]
                probs = F.softmax(logits,dim=-1)
                index_next=torch.multinomial(probs,num_samples=1)
                index = torch.cat((index,index_next),dim=1)
            return index

model = GPTLanguageModel(vocab_size)
m=model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars=decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)
                


0
 —v[—/y5L7V_yK?.O.UtN2pmC_2lu0x*v:X[MZG:;”0IS”t1.Ow0T;œ[9U(9TiOVC61œ7!:FRaoT??K7,7—W0Fqx5T,7-v“/;;R”efGoa(C—cS0FRJ6mlx,“CGœO[NMNfœ.]:VzF*ih
?6‘O?[NL_04 ‘acPe9
Pu(mX47BD’J4’o!
mEs cI51uD”“Fz6YF5gNL-m#1T—MY“FRgZ.zns.7!#/r[,vU(w7,OVr6YœJ5-rc0idLF9K*9;7—tGdWVGXHtu?Y1f1t),J[FMPYlQ4‘J5S[k,D?5a.rA!
m*(xV—v?RR[FRœCMRWsaZ”FR_?K?S.œ*d“ndn9zK2fGeSHhaGy.6Ipu
gV:w7U“qIwfU?b!Apv[EcPF5PXpXF“qUWI,KNo#*JoVq-?œ#t—v#
.œ[NJkqVZœ[œY1WtDF/Rtd(wœ,Css”G;(
.#”G:o‘[œ[s)[D1T4G0![RIFRM9N”Fl2—fel!(geOU*q57!E1Wn wSlZ:’g
en


In [22]:

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses=estimate_loss()
        print(f'step: {iter},train loss {losses['train']:.4f},val loss: {losses['val']:.4f}')
    xb,yb = get_batch('train')

    logits,loss = model.forward(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

tensor([206544, 130812,  91666,  43561, 214912,  16948,  93162,  88090, 237067,
        235322, 152479,  50244, 181536, 142462,  15964,  87005, 183161, 192002,
        192128, 170771, 182735, 155949, 215219, 158519,  13569,  38442, 234467,
         71855,  56376,  55591, 117730, 211115])
tensor([105132, 208752,  24368,  80272,  98085,  56011,  12471,  28082, 158300,
        147499,  72109,  49361, 102084, 118353,  76342,  99850, 228298,  45897,
         57374, 212810, 197672, 148120,    313,  24623, 164921, 128823, 122980,
        145061, 142412, 205238, 217591,  32870])
tensor([ 75061, 237264,  22935,   4347, 209641, 237378, 219296, 153542, 239223,
        167287, 165100, 101337, 203132, 120420, 162164, 110952, 236411,  17781,
        207430,  92736,  33744, 208371, 213029, 121777,  81625, 117338,  72426,
        234887, 210957, 237701,  55447, 197943])
tensor([181625, 147135,  95535, 150593, 108792, 146523, 139816, 146092, 117215,
         42889, 151076, 235924, 221333, 140478,  6604